# Tutorial 5: A/B Testing
## Compare a control and treatment without turning chance into a discovery

**Course:** IE 1171  
**File used:** `ab_test.csv`  
**Level 1:** Required core—clean, estimate, and interpret one A/B test  
**Level 2:** Optional deep dive—multiple testing, p-hacking, and preregistration

---

An A/B test is a randomized comparison between two versions of an experience. In this tutorial, the outcome is whether a user converted after seeing an old or new page. Claude will help draft the Python, but the human must define the experimental unit, verify the assignment, choose the outcome and analysis before looking for significance, and decide whether the estimated effect is practically meaningful.


# <img src="tutorial-icons/learning_objectives.png" alt="Learning Objectives" width="44" style="vertical-align:middle; margin-right:10px;"> Learning Objectives

By the end of this tutorial, you should be able to:

1. **Frame a valid A/B test**
   - Identify the experimental unit, control, treatment, assignment, and outcome.
   - Write the null and alternative hypotheses before analyzing results.
   - Explain when random assignment supports a causal claim.
2. **Estimate and interpret a treatment effect**
   - Audit assignment mismatches, duplicate users, missing values, and group balance.
   - Calculate conversion rates, absolute lift, relative lift, uncertainty, and a two-proportion test.
   - Separate statistical evidence from practical importance.
3. **Protect an analysis from false discoveries**
   - Explain family-wise error rate and false discovery rate.
   - Compare Bonferroni, Holm, and Benjamini–Hochberg corrections.
   - Recognize optional stopping, subgroup fishing, selective reporting, and other forms of p-hacking.


# Pólya’s Four-Step Problem-Solving Cycle

> **Backbone for this tutorial:** George Pólya’s four steps organize the work from problem framing through verification. The steps are a **cycle**, not a one-way checklist: if later evidence exposes a bad assumption, return to the earlier step that needs revision.

| Marker | Pólya step | Guiding question | In this tutorial |
|---|---|---|---|
| **🔵 🧭** | **Understand the Problem** | What is the real problem, what is known, and what constraints define success? | Define the experimental question, metric, hypothesis, and stopping rule before inspecting outcomes. |
| **🟣 🗺️** | **Devise a Plan** | What sequence of actions and checks should connect the current state to the goal? | Prespecify cleaning and analysis choices so the plan is not rewritten after seeing the result. |
| **🟠 🛠️** | **Carry Out the Plan** | Can the plan be executed in small, observable steps and checked as it runs? | Estimate conversion rates, lift, uncertainty, and other planned quantities. |
| **🟢 🔎** | **Look Back** | Does the result answer the original problem, and what should be revised or generalized? | Judge statistical and practical significance while checking multiplicity and p-hacking risks. |

The colored markers reappear at the points where each step becomes the main focus. **Human Checks support the cycle, but they are not a universal checklist:** meaningful verification depends on domain knowledge, the data-generating process, and the consequences of being wrong.


# <img src="tutorial-icons/assigned_reading.png" alt="Assigned Reading" width="44" style="vertical-align:middle; margin-right:10px;"> Assigned Reading

## Statistical Reading Used Throughout the Tutorial

**James et al., _An Introduction to Statistical Learning with Applications in Python_ (ISLP), Chapter 13**

- **Level 1:** Sections 13.1 and 13.2—hypotheses, test statistics, p-values, Type I and Type II errors, power, and the challenge of repeated tests.
- **Level 2:** Sections 13.3 and 13.4—family-wise error rate, Bonferroni and Holm procedures, false discovery rate, and Benjamini–Hochberg.

Chapter 13 provides the statistical foundation for hypothesis testing and multiple testing. The notebook supplies the e-commerce A/B context and implementation details.

Level 2 uses the same `ab_test.csv` foundation and code-generated null experiments. No additional data file is required.

## Ethical / Social-Good Reading

**Michael Kearns and Aaron Roth, _The Ethical Algorithm_**

- Chapter 1, **“Whom Do We Trust?”** (pp. 45–47)
- Chapter 1, **“Out of the Lab and Into the Wild”** (pp. 47–50)

Use these readings to connect statistical evidence to real deployment: an experiment can be internally convincing and still fail when its assumptions, incentives, or consequences change outside the test.


# <img src="tutorial-icons/tutorial_flow.png" alt="Tutorial Flow" width="44" style="vertical-align:middle; margin-right:10px;"> Tutorial Flow

| Part | Purpose |
|---|---|
| **1. Frame the experiment** | Define the unit, groups, outcome, hypotheses, and causal question. |
| **2. Audit and clean** | Resolve assignment mismatches and duplicate users before using outcomes. |
| **3. Estimate lift** | Compare conversion rates and effect sizes. |
| **4. Quantify uncertainty** | Use a confidence interval and two-proportion test. |
| **5. Level 2: Multiple testing** | Define one family and compare error-control procedures. |
| **6. Level 2: P-hacking** | Simulate optional stopping and document analyst choices. |
| **7. Preregister** | Freeze the outcome, exclusions, stopping rule, and analysis before results. |

## Human–AI rule

> Claude may calculate the evidence. The human must define the experiment before looking at that evidence and must report every analysis that belongs to the planned family.


## Tutorial Symbols

| Symbol | Meaning | What to do |
|---|---|---|
| **🔵 🧭  🟣 🗺️  🟠 🛠️  🟢 🔎** | **Pólya Backbone** | Treat the four colored checkpoints as the main problem-solving cycle; return to an earlier step when new evidence requires revision. |
| <img src="tutorial-icons/tutorial_flow.png" alt="Tutorial Flow" width="28" style="vertical-align:middle; margin-right:8px;"> | **Tutorial Flow** | Follow the notebook's normal route. |
| <img src="tutorial-icons/learning_objectives.png" alt="Learning Objectives" width="28" style="vertical-align:middle; margin-right:8px;"> | **Learning Objectives** | See the three destinations for the tutorial. |
| <img src="tutorial-icons/assigned_reading.png" alt="Assigned Reading" width="28" style="vertical-align:middle; margin-right:8px;"> | **Assigned Reading** | Read the named sections before or alongside the notebook. |
| <img src="tutorial-icons/theory.png" alt="Theory" width="28" style="vertical-align:middle; margin-right:8px;"> | **Theory** | Connect equations, assumptions, and concepts to the current part. |
| <img src="tutorial-icons/manual_pause.png" alt="Manual Pause" width="28" style="vertical-align:middle; margin-right:8px;"> | **Manual Pause** | Think or predict before asking Claude. |
| <img src="tutorial-icons/without_claude.png" alt="Without Claude" width="28" style="vertical-align:middle; margin-right:8px;"> | **Without Claude** | Notice the details an AI collaborator can coordinate. |
| <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="28" style="vertical-align:middle; margin-right:8px;"> | **Claude Task** | Use one focused and checkable prompt. |
| <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="28" style="vertical-align:middle; margin-right:8px;"> | **Your Workspace** | Paste, read, and run the response. |
| <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="28" style="vertical-align:middle; margin-right:8px;"> | **Reference Solution** | Compare only after your own attempt. |
| <img src="tutorial-icons/human_check.png" alt="Human Check" width="28" style="vertical-align:middle; margin-right:8px;"> | **Human Check — domain expertise required** | Use the provided questions, then add a domain-specific check. The notebook cannot supply a complete checklist for every application. |
| <img src="tutorial-icons/look_back.png" alt="Look Back" width="28" style="vertical-align:middle; margin-right:8px;"> | **Look Back** | Interpret, challenge, and reflect on the result. |
| <img src="tutorial-icons/level_2.png" alt="Level 2 Challenge" width="28" style="vertical-align:middle; margin-right:8px;"> | **Level 2 — Challenge Ahead** | Take the optional harder route after Level 1. |

> **Important — Human Checks are not a complete checklist.** The notebook can suggest generic verification questions, but deciding what *must* be checked depends on knowledge of the domain, the data-generating process, and the consequences of an error. If you do not have that expertise, involve someone who does. Every Human Check asks you to add your own domain-specific question.

# <img src="tutorial-icons/theory.png" alt="Theory" width="44" style="vertical-align:middle; margin-right:10px;"> Theory Foundation: Random Assignment, Estimation, and Evidence

Let $Y_i(1)$ be the outcome user $i$ would have under treatment and $Y_i(0)$ the outcome the same user would have under control. We can observe only one of these two potential outcomes for each user. Random assignment makes the observed treatment and control groups comparable on average, allowing the difference in sample conversion rates to estimate an average treatment effect.

For binary conversion,

$$
\hat p_T=\frac{x_T}{n_T},
\qquad
\hat p_C=\frac{x_C}{n_C},
\qquad
\widehat{\Delta}=\hat p_T-\hat p_C.
$$

$\widehat{\Delta}$ is the **absolute lift**, or risk difference. Relative lift is

$$
\frac{\hat p_T-\hat p_C}{\hat p_C},
$$

when $\hat p_C>0$. Absolute and relative lift answer different questions and should both include their units.

A common two-sided hypothesis test is

$$
H_0:p_T=p_C
\qquad\text{versus}\qquad
H_A:p_T\ne p_C.
$$

Under the null, the two-proportion $z$ statistic compares the observed difference with the amount of random variation expected if both groups share one conversion probability. The p-value is the probability, assuming $H_0$ and the test assumptions hold, of obtaining a test statistic at least as extreme as the observed statistic. It is not the probability that $H_0$ is true, and it does not measure effect size.

A confidence interval reports a range of effects compatible with the data and model assumptions. Statistical significance does not establish practical importance. A very large experiment can detect a tiny effect that is not worth deploying, while a small experiment can miss an important effect because it has low power.

### Questions you should be ready to answer

- What was randomized: users, sessions, devices, or page views?
- What outcome and stopping rule were chosen before results were inspected?
- What causal claim does random assignment support?
- What is the difference between absolute lift, relative lift, and a p-value?
- What decision would change if the confidence interval included zero?


## 🔵 🧭 Pólya Step 1 — Understand the Problem

**Backbone checkpoint.** State the real goal, evidence, constraints, and what would count as success before asking an AI system to solve anything.

**In this tutorial:** Define the experimental question, metric, hypothesis, and stopping rule before inspecting outcomes.

# Level 1 — Required Core

# Part 1: Frame the Experiment Before Opening the Results

This tutorial uses the Kaggle e-commerce A/B-testing file `ab_test.csv`.

| Column | Role | Meaning |
|---|---|---|
| `id` | Experimental-unit identifier | The user assigned to an experience |
| `time` | Pre-outcome context | Minute and second within the recorded hour; the file does not supply a full date or hour |
| `con_treat` | Assigned group | `control` or `treatment` |
| `page` | Experience received | `old_page` or `new_page` |
| `converted` | Binary outcome | 1 if the user converted; otherwise 0 |

The intended assignment pairs are control with `old_page` and treatment with `new_page`. Rows that violate this pairing do not represent the intended comparison and must be counted before an exclusion rule is applied.

## Confirmatory question

> Does assigning a user to the new page change the probability of conversion compared with assigning the user to the old page?

This is a causal question only if assignment was randomized, interference between users is negligible, outcomes were measured consistently, and the exclusions were not chosen after seeing which result looked favorable.


## 🟣 🗺️ Pólya Step 2 — Devise a Plan

**Backbone checkpoint.** Decide the sequence of actions and checks before the main execution. Make assumptions, evaluation rules, and stopping conditions visible so they can be challenged.

**In this tutorial:** Prespecify cleaning and analysis choices so the plan is not rewritten after seeing the result.

## <img src="tutorial-icons/manual_pause.png" alt="Manual Pause" width="36" style="vertical-align:middle; margin-right:9px;"> Manual Pause: Write the Analysis Contract

Before loading the outcome values, answer:

1. What is one experimental unit?
2. Which experience is the control and which is the treatment?
3. What exactly counts as conversion?
4. Write $H_0$ and $H_A$ in words.
5. Will the test be one-sided or two-sided, and why?
6. Which assignment mismatches will be excluded?
7. If one user appears twice, which record will be retained?
8. What sample-size or time stopping rule will be used?
9. What minimum absolute lift would matter in practice?
10. Which result will be reported if it is not statistically significant?


## 🟠 🛠️ Pólya Step 3 — Carry Out the Plan

**Backbone checkpoint.** Execute in small, observable steps. Read generated code or actions, stay within scope, and compare outputs with the behavior you predicted.

**In this tutorial:** Estimate conversion rates, lift, uncertainty, and other planned quantities.

## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 1: Load and Audit the Experiment

        ```text
        Act as a careful Python tutor.

Write one Jupyter Notebook code cell that:

1. imports pathlib, pandas, NumPy, matplotlib, seaborn, scipy.stats.norm,
   statsmodels.stats.proportion.proportions_ztest, and
   statsmodels.stats.multitest.multipletests;
2. loads ab_test.csv;
3. confirms that id, time, con_treat, page, and converted exist;
4. keeps those five columns;
5. prints the shape, first five rows, data types, and missing-value counts;
6. displays control/treatment counts, old/new page counts, and a group-by-page table;
7. counts duplicated user IDs;
8. checks that converted contains only 0 and 1;
9. does not remove rows or test an effect.

Name the dataframe ab.
Return only the Python code.
        ```

        ### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

        Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 1


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 1


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from scipy.stats import norm
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.proportion import proportions_ztest

ab_path = Path("ab_test.csv")
assert ab_path.exists(), f"File not found: {ab_path.resolve()}"

ab = pd.read_csv(ab_path)
required = ["id", "time", "con_treat", "page", "converted"]
missing_columns = sorted(set(required) - set(ab.columns))
assert not missing_columns, f"Missing required columns: {missing_columns}"
ab = ab[required].copy()

print("Shape:", ab.shape)
display(ab.head())
print("\nData types:")
display(ab.dtypes.to_frame("dtype"))
print("\nMissing values:")
display(ab.isna().sum().to_frame("missing_count"))
print("\nAssigned groups:")
display(ab["con_treat"].value_counts(dropna=False).to_frame("count"))
print("\nPages received:")
display(ab["page"].value_counts(dropna=False).to_frame("count"))
print("\nAssignment-by-page table:")
display(pd.crosstab(ab["con_treat"], ab["page"], dropna=False))

duplicate_rows = int(ab["id"].duplicated(keep=False).sum())
duplicate_users = int(ab.loc[ab["id"].duplicated(keep=False), "id"].nunique())
print(f"\nRows belonging to duplicated IDs: {duplicate_rows}")
print(f"Duplicated user IDs: {duplicate_users}")

observed_outcomes = set(ab["converted"].dropna().unique())
assert observed_outcomes <= {0, 1}, (
    f"Unexpected converted values: {sorted(observed_outcomes)}"
)


### <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


- Do the group and page labels match the codebook?
- How many rows violate the intended assignment-to-page pairing?
- Are duplicated IDs repeated observations of one user or different sessions?
- Were any exclusions made before their counts were shown?
- Does the file establish that assignment was randomized, or is that information external to the file?

# Part 2: Apply the Prespecified Cleaning Rule

The primary analysis keeps only the intended pairings:

- `control` with `old_page`;
- `treatment` with `new_page`.

It then keeps the first row for a repeated `id`. This rule provides a deterministic duplicate policy, but it must be documented because another duplicate rule could change the answer. If `time` establishes an order, a production analysis should sort by parsed time before keeping the first exposure.


## <img src="tutorial-icons/without_claude.png" alt="Without Claude" width="36" style="vertical-align:middle; margin-right:9px;"> Without Claude: Why Cleaning Can Become P-Hacking

Cleaning is necessary, but flexible cleaning creates analyst degrees of freedom. If an analyst tries “keep first,” “keep last,” “drop all duplicates,” and several time windows, then reports only the version with the smallest p-value, the cleaning process has become part of the hidden hypothesis search.

A defensible workflow:

1. defines the rule before outcomes are examined;
2. reports how many rows each rule removes;
3. keeps an audit table;
4. runs sensitivity analyses transparently rather than selecting one favorable result.


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 2: Create an Auditable Analysis Table

        ```text
        Using ab:

1. create Boolean indicators for the two intended group-page pairings;
2. count valid rows, mismatched rows, rows with missing required values,
   rows belonging to duplicated IDs, and unique duplicated IDs;
3. display these counts in one audit table;
4. retain only valid pairings with complete required values;
5. keep the first row for each id and state this rule in a comment;
6. assert that every remaining ID is unique;
7. assert that control maps only to old_page and treatment only to new_page;
8. name the cleaned dataframe ab_clean;
9. do not calculate conversion rates yet.

Return only the Python code.
        ```

        ### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

        Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 2


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 2


In [ ]:
intended_pair = (
    ((ab["con_treat"] == "control") & (ab["page"] == "old_page"))
    | ((ab["con_treat"] == "treatment") & (ab["page"] == "new_page"))
)
complete_required = ab[
    ["id", "time", "con_treat", "page", "converted"]
].notna().all(axis=1)
duplicated_id_row = ab["id"].duplicated(keep=False)

audit = pd.DataFrame(
    {
        "check": [
            "all rows",
            "valid group-page pairing",
            "mismatched group-page pairing",
            "rows missing a required value",
            "rows belonging to duplicated IDs",
            "unique duplicated IDs",
        ],
        "count": [
            len(ab),
            int(intended_pair.sum()),
            int((~intended_pair).sum()),
            int((~complete_required).sum()),
            int(duplicated_id_row.sum()),
            int(ab.loc[duplicated_id_row, "id"].nunique()),
        ],
    }
)
display(audit)

# Prespecified duplicate rule: after valid-pair and completeness filters,
# keep the first observed row for each experimental-unit ID.
ab_clean = (
    ab.loc[intended_pair & complete_required]
    .drop_duplicates(subset="id", keep="first")
    .copy()
)
ab_clean["converted"] = ab_clean["converted"].astype(int)

assert ab_clean["id"].is_unique
assert set(ab_clean.loc[ab_clean["con_treat"] == "control", "page"]) == {
    "old_page"
}
assert set(
    ab_clean.loc[ab_clean["con_treat"] == "treatment", "page"]
) == {"new_page"}

print("Clean analysis shape:", ab_clean.shape)


### <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


1. Reconcile the original row count with the exclusions and final count.
2. Confirm that the outcome was not used to decide which duplicates to keep.
3. Explain why assignment mismatch is different from treatment noncompliance.
4. Decide whether the exclusion rule estimates the effect of assignment or the effect among users who received the intended page.
5. Record the exact cleaning rule in your analysis notes.

# Part 3: Estimate Conversion Rates and Lift

The point estimate should come before the significance label. Report:

- control conversion rate;
- treatment conversion rate;
- absolute lift in percentage points;
- relative lift as a percentage of the control rate;
- group sizes and conversion counts.

A difference of `0.002` means 0.2 percentage points, not 0.2 percent. Clear units prevent a small number from being made to sound larger than it is.


## <img src="tutorial-icons/theory.png" alt="Theory" width="36" style="vertical-align:middle; margin-right:9px;"> Theory: Effect Size Comes Before the Decision

If the control conversion rate is 12% and the treatment rate is 12.6%:

- absolute lift is $12.6\%-12.0\%=0.6$ percentage points;
- relative lift is $0.6/12.0=5\%$.

Neither number says whether the result is due to chance. The p-value and confidence interval address uncertainty; the effect size addresses magnitude. A deployment decision may also require revenue, cost, risk, duration, and distributional effects.


## <img src="tutorial-icons/manual_pause.png" alt="Manual Pause" width="36" style="vertical-align:middle; margin-right:9px;"> Manual Pause: Predict the Result Table

Before running the calculation:

1. Which row should be treated as the baseline?
2. What unit will be used for absolute lift?
3. What denominator appears in relative lift?
4. What happens to relative lift if the control rate is zero?
5. Would an unbalanced group size invalidate the experiment automatically?
6. What imbalance would make you investigate the assignment system?


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 3: Estimate Conversion Lift

        ```text
        Using ab_clean:

1. calculate for each con_treat group the number of users, conversions,
   non-conversions, and conversion rate;
2. calculate treatment minus control absolute lift;
3. report absolute lift in probability units and percentage points;
4. calculate relative lift only when the control rate is greater than zero;
5. create a two-bar conversion-rate plot with the y-axis beginning at zero;
6. label each bar with the rate and sample size;
7. do not run a hypothesis test or call either page better.

Return only the Python code.
        ```

        ### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

        Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 3


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 3


In [ ]:
group_summary = (
    ab_clean.groupby("con_treat")["converted"]
    .agg(users="size", conversions="sum", conversion_rate="mean")
    .reindex(["control", "treatment"])
)
group_summary["non_conversions"] = (
    group_summary["users"] - group_summary["conversions"]
)
group_summary = group_summary[
    ["users", "conversions", "non_conversions", "conversion_rate"]
]
display(group_summary)

control_rate = float(group_summary.loc["control", "conversion_rate"])
treatment_rate = float(group_summary.loc["treatment", "conversion_rate"])
absolute_lift = treatment_rate - control_rate
relative_lift = (
    absolute_lift / control_rate if control_rate > 0 else np.nan
)

print(f"Absolute lift: {absolute_lift:.6f}")
print(f"Absolute lift: {100 * absolute_lift:.3f} percentage points")
print(f"Relative lift: {100 * relative_lift:.3f}%")

ax = group_summary["conversion_rate"].plot(
    kind="bar", color=["#4C78A8", "#F58518"], rot=0, figsize=(7, 4)
)
ax.set_ylim(bottom=0)
ax.set_ylabel("Conversion rate")
ax.set_xlabel("Assigned group")
ax.set_title("Observed conversion rate by assigned group")
for position, (_, row) in enumerate(group_summary.iterrows()):
    ax.text(
        position,
        row["conversion_rate"],
        f'{row["conversion_rate"]:.3%}\nn={int(row["users"]):,}',
        ha="center",
        va="bottom",
    )
plt.tight_layout()
plt.show()


### <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


- Are treatment and control in the intended order?
- Is the effect treatment minus control rather than the reverse?
- Are percentages and percentage points distinguished?
- Does the chart's vertical scale make the difference look larger than it is?
- Is the observed lift large enough to matter under the rule written before analysis?

# Part 4: Quantify Uncertainty

The primary analysis uses a two-sided two-proportion $z$ test and a 95% confidence interval for treatment minus control. The test and interval answer related questions:

- the p-value measures compatibility with the no-difference null;
- the interval shows the range of effect sizes compatible with the data under the approximation;
- neither decides whether the page should be deployed.

The normal interval used here is transparent and appropriate for a large experiment with enough conversions in each group. Small counts would require a different interval or an exact/randomization approach.


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 4: Test the Prespecified Effect

        ```text
        Using group_summary:

1. test H0: p_treatment = p_control with a two-sided proportions_ztest;
2. calculate a 95% normal-approximation confidence interval for
   treatment minus control;
3. print the z statistic, p-value, point estimate, and confidence interval;
4. print whether the interval includes zero;
5. calculate expected conversions per 10,000 users under each observed rate;
6. use the phrase fail to reject rather than accept the null;
7. do not make a deployment recommendation.

Return only the Python code.
        ```

        ### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

        Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 4


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 4


In [ ]:
successes = np.array(
    [
        group_summary.loc["treatment", "conversions"],
        group_summary.loc["control", "conversions"],
    ],
    dtype=float,
)
observations = np.array(
    [
        group_summary.loc["treatment", "users"],
        group_summary.loc["control", "users"],
    ],
    dtype=float,
)

z_statistic, p_value = proportions_ztest(
    count=successes, nobs=observations, value=0, alternative="two-sided"
)

treatment_se_term = treatment_rate * (1 - treatment_rate) / observations[0]
control_se_term = control_rate * (1 - control_rate) / observations[1]
difference_se = np.sqrt(treatment_se_term + control_se_term)
critical_value = norm.ppf(0.975)
ci_low = absolute_lift - critical_value * difference_se
ci_high = absolute_lift + critical_value * difference_se

print(f"z statistic: {z_statistic:.4f}")
print(f"two-sided p-value: {p_value:.6g}")
print(f"treatment - control: {absolute_lift:.6f}")
print(f"95% CI: [{ci_low:.6f}, {ci_high:.6f}]")
print("Interval includes zero:", ci_low <= 0 <= ci_high)
print(
    "Observed conversions per 10,000 users — "
    f"control: {10_000 * control_rate:.1f}, "
    f"treatment: {10_000 * treatment_rate:.1f}"
)

if p_value < 0.05:
    print("Decision at alpha=0.05: reject the no-difference null.")
else:
    print("Decision at alpha=0.05: fail to reject the no-difference null.")


### <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


1. Does the confidence interval use treatment minus control?
2. Is zero inside the interval?
3. Which effects are compatible with the interval but practically unimportant?
4. Which practically important effects remain possible?
5. Are there enough conversions and non-conversions for the approximation?
6. What assumption outside the CSV is necessary for a causal interpretation?
7. Would you report the same result if the new page appeared worse?

## 🟢 🔎 Pólya Step 4 — Look Back

**Backbone checkpoint.** Do not stop at “it ran.” Ask whether the result answers the original problem, what evidence supports it, what failed, and what should change. Domain expertise matters here because a generic checklist cannot know every real-world failure mode.

**In this tutorial:** Judge statistical and practical significance while checking multiplicity and p-hacking risks.

### <img src="tutorial-icons/look_back.png" alt="Look Back" width="30" style="vertical-align:middle; margin-right:8px;"> Level 1 Look Back

Write a four-sentence result:

1. Describe the experimental comparison and cleaned sample.
2. Report both group rates and treatment-minus-control lift.
3. Report the confidence interval and p-value without saying the null is true.
4. State one design or measurement limitation that the calculation cannot resolve.


# AI for Social Good: Experiments Leave the Lab and Change People’s Experience

A/B testing can help public-interest organizations evaluate messages, volunteer outreach, health reminders, accessibility changes, or service-delivery designs. A responsible deployment loop tests an intervention, observes unintended consequences, revises it, and only then considers broader rollout.

Randomization improves causal evidence, but it does **not** answer every ethical question. Before deployment, ask:

- Is it acceptable to randomize this treatment, and are risks minimal?
- Is the measured outcome actually connected to the social goal, or only easy to optimize?
- Could a treatment help one group while burdening another?
- Are short-term clicks or conversions replacing long-term well-being?
- Were negative and null results documented rather than hidden?
- Are repeated tests creating false discoveries that could drive policy or product changes?
- What should happen if the experiment causes an unexpected harm after rollout?

The multiple-testing lesson is especially important for social-good work: searching many outcomes or subgroups until something is “significant” can create confident stories that do not replicate in the real world.

> **Social-good principle:** Randomization strengthens evidence about an intervention; it does not automatically make the intervention, outcome, or rollout responsible.


# Tutorial 5 Conclusion

You used Claude to audit an experiment, preserve a documented cleaning rule, compare control and treatment conversion rates, calculate absolute and relative lift, and quantify uncertainty with a confidence interval and two-proportion test.

The main lesson is that an A/B test is not a search for a p-value below 0.05. It is a prespecified comparison whose design, effect size, uncertainty, and consequences must be interpreted together.


# <img src="tutorial-icons/look_back.png" alt="Look Back" width="44" style="vertical-align:middle; margin-right:10px;"> Final Reflection

1. What is the experimental unit?
2. What do the control, treatment, and outcome represent?
3. Why must exclusions be fixed before comparing outcomes?
4. What is the difference between absolute and relative lift?
5. What does the p-value mean under the null hypothesis?
6. What does it not mean?
7. Why does a confidence interval add information beyond a significance label?
8. What allows a causal interpretation?
9. Why can a statistically significant effect be unimportant?
10. Which experimental judgment could not be delegated to Claude?


# <img src="tutorial-icons/level_2.png" alt="Level 2 Challenge" width="44" style="vertical-align:middle; margin-right:10px;"> Level 2 — Optional Deep Dive

> **Challenge ahead:** Complete Level 1 first. Level 2 keeps the same experiment and reading foundation, then examines what happens when an analyst tests many ideas or repeatedly checks the same experiment.

# Part 5: Multiple Testing

A **family** is the set of hypotheses that belong to one scientific or decision-making claim. If an analyst tests dayparts, weekdays, devices, countries, outcomes, and several cleaning rules, the relevant family is not only the one favorable test eventually reported.

For $m$ independent true null hypotheses tested at per-test level $\alpha$,

$$
P(\text{at least one false positive})=1-(1-\alpha)^m.
$$

At $\alpha=0.05$ and $m=20$, this is about 64%, not 5%.


## <img src="tutorial-icons/theory.png" alt="Theory" width="36" style="vertical-align:middle; margin-right:9px;"> Theory: FWER and FDR Control Different Risks

**Family-wise error rate (FWER)** is the probability of making at least one Type I error in the family. Bonferroni rejects a test only when $p_j\le\alpha/m$. Holm improves power while still controlling FWER.

**False discovery rate (FDR)** is the expected fraction of rejected hypotheses that are false discoveries. Benjamini–Hochberg controls FDR under the conditions described in ISLP and is often more useful for large exploratory families.

These procedures answer different questions:

- use FWER control when even one false claim is costly;
- use FDR control when a larger exploratory set will receive independent follow-up;
- do not choose the correction after seeing which one preserves significance.


## <img src="tutorial-icons/manual_pause.png" alt="Manual Pause" width="36" style="vertical-align:middle; margin-right:9px;"> Manual Pause: Define the Family

The downloaded file stores `time` as minute and second within an hour (for example, `11:48.6`), not as a full timestamp. It therefore cannot support honest daypart or weekday comparisons. The next example instead uses six prespecified minute-within-hour comparisons:

- four non-overlapping 15-minute quarters: 00–14, 15–29, 30–44, and 45–59;
- two non-overlapping 30-minute halves: 00–29 and 30–59.

Before running them:

1. Why do these six tests form one family?
2. Which comparisons overlap?
3. What would one false positive mean?
4. Would FWER or FDR match a confirmatory deployment decision?
5. Why must the segment definitions be written before the p-values are seen?


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 5: Compare Raw and Adjusted Segment Tests

        ```text
        Using ab_clean:

1. parse the MM:SS.s time strings into minute and second fields and report failed parses;
2. create four prespecified 15-minute quarters and two 30-minute halves;
3. run a two-sided two-proportion z-test within each of the six segments;
4. store segment name, treatment and control sample sizes, both rates,
   treatment-minus-control lift, and raw p-value;
5. adjust the six p-values with Bonferroni, Holm, and Benjamini-Hochberg;
6. show raw and adjusted reject decisions at 0.05;
7. sort by segment definition order, not by smallest p-value;
8. report every test, including null results.

Return only the Python code.
        ```

        ### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

        Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 5


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 5


In [ ]:
ab_level2 = ab_clean.copy()
time_parts = ab_level2["time"].astype("string").str.extract(
    r"^(?P<minute>\d{2}):(?P<second>\d{2}\.\d)$"
)
failed_time_parses = int(time_parts.isna().any(axis=1).sum())
print("Failed time parses:", failed_time_parses)

valid_time = ~time_parts.isna().any(axis=1)
ab_level2 = ab_level2.loc[valid_time].copy()
ab_level2["minute"] = time_parts.loc[valid_time, "minute"].astype(int)
ab_level2["second"] = time_parts.loc[valid_time, "second"].astype(float)
assert ab_level2["minute"].between(0, 59).all()
assert ab_level2["second"].between(0, 60, inclusive="left").all()

ab_level2["minute_quarter"] = pd.cut(
    ab_level2["minute"],
    bins=[-1, 14, 29, 44, 59],
    labels=["00-14", "15-29", "30-44", "45-59"],
)
ab_level2["minute_half"] = np.where(
    ab_level2["minute"] < 30, "00-29", "30-59"
)

family = [
    ("minute-quarter: 00-14", ab_level2["minute_quarter"] == "00-14"),
    ("minute-quarter: 15-29", ab_level2["minute_quarter"] == "15-29"),
    ("minute-quarter: 30-44", ab_level2["minute_quarter"] == "30-44"),
    ("minute-quarter: 45-59", ab_level2["minute_quarter"] == "45-59"),
    ("minute-half: 00-29", ab_level2["minute_half"] == "00-29"),
    ("minute-half: 30-59", ab_level2["minute_half"] == "30-59"),
]

segment_rows = []
for segment_name, mask in family:
    segment = ab_level2.loc[mask]
    by_group = segment.groupby("con_treat")["converted"].agg(
        ["size", "sum", "mean"]
    )
    if not {"control", "treatment"} <= set(by_group.index):
        raise ValueError(f"Both groups are not present in {segment_name}")

    counts = np.array(
        [
            by_group.loc["treatment", "sum"],
            by_group.loc["control", "sum"],
        ]
    )
    nobs = np.array(
        [
            by_group.loc["treatment", "size"],
            by_group.loc["control", "size"],
        ]
    )
    z_value, raw_p = proportions_ztest(counts, nobs, alternative="two-sided")
    segment_rows.append(
        {
            "segment": segment_name,
            "n_treatment": int(nobs[0]),
            "n_control": int(nobs[1]),
            "rate_treatment": float(by_group.loc["treatment", "mean"]),
            "rate_control": float(by_group.loc["control", "mean"]),
            "lift": float(
                by_group.loc["treatment", "mean"]
                - by_group.loc["control", "mean"]
            ),
            "raw_p": float(raw_p),
        }
    )

segment_tests = pd.DataFrame(segment_rows)
for method, short_name in [
    ("bonferroni", "bonferroni"),
    ("holm", "holm"),
    ("fdr_bh", "bh_fdr"),
]:
    reject, adjusted_p, _, _ = multipletests(
        segment_tests["raw_p"], alpha=0.05, method=method
    )
    segment_tests[f"{short_name}_p"] = adjusted_p
    segment_tests[f"{short_name}_reject"] = reject

segment_tests["raw_reject"] = segment_tests["raw_p"] < 0.05
display(segment_tests)


### <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


- Did the code report all six planned tests?
- Which raw decisions changed after adjustment?
- Which method controls the chance of any false positive?
- Which method controls the expected proportion of false discoveries?
- Were the segments defined using pre-treatment information?
- Are the subgroup estimates precise enough to interpret?
- Would the same family be reported if no segment were significant?

# Part 6: P-Hacking and Optional Stopping

**P-hacking** means using analysis flexibility in a way that makes a result look more convincing than the full process supports. It can be intentional, but it can also arise from ordinary human confirmation bias.

Common paths include:

- repeatedly checking the p-value and stopping when it falls below 0.05;
- trying many subgroups but reporting only one;
- changing exclusions after seeing outcomes;
- swapping outcomes or test directions;
- trying several model specifications and presenting one as planned;
- treating exploratory analysis as confirmatory.

The next simulation has no treatment effect. Both groups convert at the same 12% rate. The only change is that the analyst looks several times and stops after the first significant result.


## <img src="tutorial-icons/theory.png" alt="Theory" width="36" style="vertical-align:middle; margin-right:9px;"> Theory: Repeated Looks Are Repeated Opportunities

A fixed-sample p-value controls the Type I error for the prespecified test at the prespecified sample size. It does not automatically control the error rate for “look after 200 users, then 400, then 800, and stop whenever $p<0.05$.”

Sequential experiments can be valid, but they require a planned sequential method, alpha-spending rule, always-valid inference, or another design that accounts for repeated looks. The problem is hidden flexibility, not the existence of interim analysis.


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Coding Task 6: Simulate Optional Stopping Under No Effect

        ```text
        Write one Jupyter code cell that:

1. uses np.random.default_rng(1099);
2. simulates 500 experiments with no treatment effect;
3. assigns users randomly to control or treatment;
4. gives every user the same 0.12 conversion probability;
5. checks a two-sided proportions z-test after total sample sizes
   200, 400, 800, 1200, and 2000;
6. records whether any look has p < 0.05 and the first significant look;
7. reports the fixed-final-look false-positive rate and the
   stop-when-significant false-positive rate;
8. plots the cumulative fraction of experiments with at least one
   significant result by look;
9. labels this as a null simulation, not evidence about ab_test.csv.

Return only the Python code.
        ```

        ### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

        Paste, read, and run Claude's response in the next cell.


In [ ]:
# YOUR WORKSPACE — Claude Coding Task 6


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 6


In [ ]:
rng = np.random.default_rng(1099)
looks = np.array([200, 400, 800, 1200, 2000])
simulations = 500
p_values_by_look = np.full((simulations, len(looks)), np.nan)

for simulation in range(simulations):
    assigned_treatment = rng.integers(0, 2, size=looks[-1])
    converted_null = rng.binomial(1, 0.12, size=looks[-1])

    for look_index, sample_size in enumerate(looks):
        assigned_now = assigned_treatment[:sample_size]
        converted_now = converted_null[:sample_size]
        treatment_now = converted_now[assigned_now == 1]
        control_now = converted_now[assigned_now == 0]

        counts_now = np.array(
            [treatment_now.sum(), control_now.sum()], dtype=float
        )
        nobs_now = np.array(
            [len(treatment_now), len(control_now)], dtype=float
        )
        _, p_now = proportions_ztest(
            counts_now, nobs_now, alternative="two-sided"
        )
        p_values_by_look[simulation, look_index] = p_now

significant_by_look = p_values_by_look < 0.05
fixed_final_false_positive = significant_by_look[:, -1].mean()
optional_stopping_false_positive = significant_by_look.any(axis=1).mean()
cumulative_any = np.maximum.accumulate(significant_by_look, axis=1).mean(axis=0)

first_significant_look = []
for row in significant_by_look:
    significant_indices = np.flatnonzero(row)
    first_significant_look.append(
        int(looks[significant_indices[0]])
        if len(significant_indices)
        else np.nan
    )

print("Null simulation: both groups convert at 12%.")
print(
    "False-positive rate using only the fixed final look:",
    f"{fixed_final_false_positive:.3f}",
)
print(
    "False-positive rate when stopping after any significant look:",
    f"{optional_stopping_false_positive:.3f}",
)
display(pd.Series(first_significant_look, name="first_significant_n").value_counts(
    dropna=False
).sort_index())

plt.figure(figsize=(7, 4))
plt.plot(looks, cumulative_any, marker="o", color="#E45756")
plt.axhline(0.05, color="black", linestyle="--", label="Nominal 0.05")
plt.xlabel("Total sample size at interim look")
plt.ylabel("Fraction with at least one p < 0.05")
plt.title("Null simulation: optional stopping inflates false positives")
plt.legend()
plt.tight_layout()
plt.show()


### <img src="tutorial-icons/human_check.png" alt="Human Check" width="30" style="vertical-align:middle; margin-right:8px;"> Human Check

> **Domain expertise required:** The questions below are examples, not a complete checklist. Add a check based on the real application; if you lack that expertise, involve someone who has it.

- **Your own domain question:** What could be wrong here that a generic AI system or checklist would be unlikely to notice?


1. Is the true treatment effect exactly zero in the simulation?
2. Why is the final-look false-positive rate near 0.05?
3. Why is the any-look rate larger?
4. Would changing the random seed remove the general pattern?
5. How could a valid sequential design preserve interim monitoring?
6. Why is “we stopped when the result was clear” not a complete stopping rule?

# Part 7: Preregister the Analysis

Preregistration records the confirmatory plan before the results are inspected. It does not prohibit exploration. It separates:

- **confirmatory work**, which follows the frozen plan;
- **exploratory work**, which generates new hypotheses and is labeled as such;
- **replication**, which tests those new hypotheses on new data.


## <img src="tutorial-icons/claude_task.png" alt="Claude Task" width="36" style="vertical-align:middle; margin-right:9px;"> Claude Collaboration Task 7: Draft a One-Page Preregistration

        ```text
        Draft a concise preregistration for the ab_test.csv experiment.

Include:

1. research question and causal estimand;
2. experimental unit;
3. control and treatment definitions;
4. primary outcome;
5. null and alternative hypotheses;
6. two-sided alpha level;
7. assignment-mismatch, missing-value, and duplicate-ID rules;
8. fixed stopping rule;
9. primary effect measure, confidence interval, and hypothesis test;
10. the six Level 2 segment tests as one family and the chosen correction;
11. a promise to report null and negative results;
12. a boundary between confirmatory and exploratory analysis.

Do not invent a business deployment threshold. Mark it as a required
decision to be supplied by the experiment owner before analysis.
        ```

        ### <img src="tutorial-icons/your_workspace.png" alt="Your Workspace" width="30" style="vertical-align:middle; margin-right:8px;"> Your Workspace

        Write Claude's draft below, then revise it yourself before comparing with the reference.


**Your revised preregistration:**

_Write here._


### <img src="tutorial-icons/reference_solution.png" alt="Reference Solution" width="30" style="vertical-align:middle; margin-right:8px;"> Reference Solution for Task 7


A complete reference plan should contain:

- **Question:** the effect of assignment to the new page on user conversion.
- **Unit:** one unique user ID.
- **Groups:** control/old page and treatment/new page.
- **Outcome:** the prespecified binary `converted` field.
- **Primary estimand:** treatment-minus-control conversion probability.
- **Primary inference:** two-sided two-proportion test and 95% confidence interval at a fixed final sample.
- **Exclusions:** incomplete required rows and invalid assignment-page pairs; keep the first valid row per duplicated ID, all fixed before outcome comparison.
- **Stopping:** a fixed sample size or end time chosen before outcome inspection.
- **Multiplicity:** the six named minute-within-hour tests are one family; use the chosen FWER or FDR procedure and report all six.
- **Reporting:** publish effect sizes, intervals, p-values, exclusions, null results, and deviations from plan.
- **Exploration:** label unplanned subgroups and specifications as exploratory and require new data for confirmation.
- **Practical threshold:** supplied and justified by the experiment owner before analysis rather than inferred from statistical significance.


# <img src="tutorial-icons/look_back.png" alt="Look Back" width="44" style="vertical-align:middle; margin-right:10px;"> Level 2 Look Back

1. What collection of tests formed the family?
2. How do Bonferroni, Holm, and Benjamini–Hochberg control different risks?
3. Why can raw subgroup p-values be misleading?
4. How did optional stopping change the false-positive rate under no effect?
5. Name three analyst choices that could become p-hacking.
6. Why can p-hacking occur without deliberate fraud?
7. What belongs in a preregistration?
8. How should an unplanned but interesting pattern be reported?
9. Why are null results part of the scientific record?
10. Which part of the workflow most needs human accountability?


# Sources and Course Resources

- James, Gareth, Daniela Witten, Trevor Hastie, and Robert Tibshirani. _An Introduction to Statistical Learning with Applications in Python_, Chapter 13, Sections 13.1–13.4.
- A/B dataset source: [Kaggle A/B Testing](https://www.kaggle.com/datasets/zhangluyuan/ab-testing).
- Harvard Business Review. [A Refresher on A/B Testing](https://hbr.org/2017/06/a-refresher-on-ab-testing).
